In [2]:
import pandas as pd
import pickle
import math
from collections import defaultdict

# ================================
# LOAD DATA
# ================================
df = pd.read_csv("s2d.csv")

df.columns = [c.strip().lower() for c in df.columns]

disease_col = df.columns[0]
symptom_cols = df.columns[1:]

print("Disease column:", disease_col)
print("Total symptoms:", len(symptom_cols))

# ================================
# CREATE DID
# ================================
diseases = df[disease_col].unique()

did = {}
id_to_disease = {}

for i, d in enumerate(diseases):
    d_clean = d.strip().lower()
    d_id = f"D_{i}"
    did[d_clean] = d_id
    id_to_disease[d_id] = d_clean

# ================================
# CREATE SID
# ================================
sid = {}
id_to_symptom = {}

for i, s in enumerate(symptom_cols):
    s_clean = s.strip().lower()
    s_id = f"S_{i}"
    sid[s_clean] = s_id
    id_to_symptom[s_id] = s_clean

# ================================
# GROUP BY DISEASE
# ================================
print("Grouping data by disease...")

grouped = df.groupby(disease_col)

# ================================
# DISEASE-LEVEL DF (IMPORTANT FIX)
# ================================
print("Computing disease-level IDF...")

# For each disease, whether symptom appears at least once
disease_symptom_matrix = grouped[symptom_cols].max()

total_diseases = len(disease_symptom_matrix)

symptom_df = {}

for s in symptom_cols:
    # number of diseases where symptom appears
    symptom_df[s] = disease_symptom_matrix[s].sum()

# compute IDF (disease-level)
symptom_idf = {}

for s in symptom_cols:
    df_s = symptom_df[s]

    idf = math.log((1 + total_diseases) / (1 + df_s)) + 1
    symptom_idf[s] = idf

# ================================
# BUILD KG
# ================================
print("Building KG...")

sd_graph = defaultdict(dict)

for d in diseases:

    d_clean = d.strip().lower()
    d_id = did[d_clean]

    subset = df[df[disease_col] == d]
    total_cases = len(subset)

    for s in symptom_cols:

        s_clean = s.strip().lower()
        s_id = sid[s_clean]

        count = subset[s].sum()

        # 🔥 smoothed probability
        p_s_given_d = (count + 1) / (total_cases + 2)

        # 🔥 disease-level IDF
        idf = symptom_idf[s]

        # 🔥 final weight
        weight = p_s_given_d * idf

        # 🔥 remove weak signals
        if weight > 0.05:
            sd_graph[s_id][d_id] = float(weight)

# ================================
# NORMALIZE PER SYMPTOM
# ================================
print("Normalizing KG...")

for s in sd_graph:
    total = sum(sd_graph[s].values()) + 1e-8
    for d in sd_graph[s]:
        sd_graph[s][d] /= total

# ================================
# SAVE FILES
# ================================
with open("SID.p", "wb") as f:
    pickle.dump(sid, f)

with open("DID.p", "wb") as f:
    pickle.dump(did, f)

with open("SD++.p", "wb") as f:
    pickle.dump(dict(sd_graph), f)

print("\n✅ FINAL KG BUILT SUCCESSFULLY")
print(f"Diseases: {len(did)}")
print(f"Symptoms: {len(sid)}")

Disease column: diseases
Total symptoms: 377
Grouping data by disease...
Computing disease-level IDF...
Building KG...
Normalizing KG...

✅ FINAL KG BUILT SUCCESSFULLY
Diseases: 773
Symptoms: 377
